# TP2 — Diseño de Arquitectura de Datos para un Medio Internacional de Noticias (OLTP + OLAP) sobre GDELT 2.0

**IMPORTANTE:** Al utilizar herramientas de inteligencia artificial deben indicar qué IA están usando y con qué propósito. Por ejemplo: "Claude Opus 4.6" para "mejorar redacción de textos" o "Chat GPT 5.6" para "crear código fuente".

En el TP1 procesaron el flujo de datos de GDELT 2.0. Ahora, la dirección de un Medio Internacional de Noticias necesita modernizar su plataforma de datos para dos propósitos clave:

- **Operación Periodística (OLTP):** Registrar en tiempo real las noticias publicadas, las fuentes consultadas, los actores involucrados y el seguimiento de eventos en desarrollo.
- **Inteligencia de Contenidos (OLAP):** Analizar el impacto, la cobertura global y las tendencias mediáticas para tomar decisiones editoriales y de distribución en tiempo real.

## Recursos

- Lista de URLs con actualizaciones cada 15 minutos: https://data.gdeltproject.org/gdeltv2/masterfilelist.txt
- URL oficial del libro de códigos de GDELT 2.0: https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
- URL de la documentación en texto/HTML donde se enuncian los campos (CAMEO Manual): https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf

## Requisitos

El diseño del Almacén de Datos debe estar estructurado de forma tal que permita responder estas preguntas clave del negocio periodístico:

1. **Rendimiento e Impacto:** ¿Cuáles son las noticias y temas que generan el mayor volumen de menciones y publicaciones (`NumMentions` / `NumArticles`) por país y mes?
2. **Sentimiento de la Cobertura:** ¿Cómo varía el tono promedio de la cobertura (`AvgTone`) sobre eventos globales según el tipo de actor involucrado (ej. Gobierno, ONG, Empresas)?
3. **Análisis de Frecuencia y Temporalidad:** ¿Cómo cambia la intensidad del flujo de noticias entre días hábiles, fines de semana y eventos festivos?
4. **Cobertura Multi-Temática (Múltiples Categorías CAMEO):** ¿Qué eventos periodísticos requirieron clasificación bajo múltiples códigos de acción o temáticas simultáneas en una misma cobertura?

## Actividades

**1)** Diseñar la estructura de la base de datos operativa basándose en el estándar CAMEO/GDELT. Presentar un diagrama Entidad-Relación con las entidades principales (Noticia_Evento, Actor, Ubicación, Fuente_Medio, Código_CAMEO), identificar llaves (PK, FK, compuesta, sustituta/surrogate) justificando su uso frente a la llave natural (`GlobalEventID`), e indicar el tipo de relaciones (1:1, 1:N, M:N, opcional, obligatoria).

**2)** Diseñar el modelo dimensional (Esquema Estrella) para la tabla de hechos del análisis de medios, con la dimensión temporal (Año, Trimestre, Mes, Día_Semana, Es_Fin_De_Semana, Es_Festivo) y una Tabla Puente (Bridge Table) para resolver la relación de una noticia/evento con múltiples códigos de categorías temáticas CAMEO.

**3)** *(Opcional)* Implementar ambas bases de datos (transaccional y almacén) mediante consultas SQL, con código ejecutable para reproducir los modelos.

## Formato de entrega

- Archivo de Draw.io en formato XML con el nombre `TP2` dentro del repositorio GitHub (diagrama ER OLTP + Esquema Estrella OLAP).
- Notebook `TP2.ipynb` con la consigna al inicio.

---

**Autora:** Aylin Agatha Vazquez Chenlo — Doctorado en Ingeniería, ITBA (bioingeniera)

**Profesor:** Nicolás Guillermo Tripp

**Repositorio:** https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos

**Diagrama Draw.io (XML):** [`TP2.drawio`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP2.drawio) — contiene 2 páginas: el modelo Entidad-Relación OLTP y el Esquema Estrella OLAP.

**Utilización de IA para co-desarrollo:** Se utilizó **Claude Sonnet 5** (Anthropic) para

- Co-generar el código SQL/Python (SQLite) de la implementación opcional (Actividad 3) y mejorar las consultas de ejemplo para las 4 preguntas de negocio.
- Corregir la redacción de las explicaciones/justificaciones en las celdas markdown del notebook.

---

## 0) Informacion importante de GDELT 2.0 y CAMEO

> Seccion en la que Claude ayudo a entender los dos documentos de referencia del enunciado. Se resume solo lo relevante para justificar las decisiones de modelado de las secciones 1) y 2).

### 0.1) GDELT Event Codebook V2.0 — estructura del feed de eventos

GDELT ("Global Database of Events, Language, and Tone") monitorea noticias en más de 100 idiomas y codifica cada interacción relatada entre dos actores (`Actor1` hace algo sobre `Actor2`) usando la taxonomía **CAMEO**. Desde la versión 2.0 (19/02/2015), el feed se genera **cada 15 minutos** (antes era diario) y se agregan dos tablas nuevas: la tabla de **Mentions** (registra cada mención del evento en cada artículo, con su nivel de `Confidence`) y el **GKG** (Global Knowledge Graph, con temas/emociones enriquecidas). Los campos clave del **Event Table** que usamos para el modelo OLTP son:

| Bloque | Campos | Para qué sirve |
|---|---|---|
| **Identificación y fecha** | `GlobalEventID`, `Day`/`MonthYear`/`Year`, `FractionDate` | `GlobalEventID` es único mundialmente, pero **no debe usarse para ordenar por fecha** (hay saltos grandes, ej. el reseteo al pasar de GDELT 1.0 a 2.0); para eso están los campos de fecha. `DATEADDED` (al final del registro) es el único con resolución de 15 minutos. |
| **Actor1 / Actor2** | `ActorCode` (código CAMEO crudo completo), `ActorName`, `ActorCountryCode`, `ActorKnownGroupCode`, `ActorEthnicCode`, `ActorReligion1/2Code`, `ActorType1/2/3Code` | Cada actor puede tener **hasta 3 roles simultáneos** (`Type1/2/3Code`, ver 0.2). Un evento puede tener Actor2 vacío (acción unilateral, ej. "gunmen" sin identificar). |
| **Acción / clasificación** | `EventCode`, `EventBaseCode`, `EventRootCode`, `QuadClass`, `GoldsteinScale` | `EventCode` es el código CAMEO de 3 dígitos más específico; `EventBaseCode` (2 niveles) y `EventRootCode` (1 nivel, la categoría 01-20) permiten agregar a distinta granularidad. **Deben guardarse como texto**, no como entero (hay códigos con ceros a la izquierda). `QuadClass` resume todo en 4 grandes clases: 1=Cooperación Verbal, 2=Cooperación Material, 3=Conflicto Verbal, 4=Conflicto Material. `GoldsteinScale` (-10 a +10) es un puntaje fijo **por tipo de evento** (no por instancia), que mide su impacto teórico sobre la estabilidad. |
| **Cobertura mediática** | `IsRootEvent`, `NumMentions`, `NumSources`, `NumArticles`, `AvgTone` | Miden la "importancia" del evento: cuántas veces se mencionó, en cuántas fuentes/artículos distintos y el tono promedio del texto (-100 a +100, valores típicos entre -10 y +10). GDELT aclara que estos campos son de compatibilidad histórica y que la tabla `Mentions` es la fuente más precisa — pero alcanzan para nuestras 4 preguntas de negocio. |
| **Geografía** | `Actor1Geo_*`, `Actor2Geo_*`, `ActionGeo_*` (cada uno con `Type`, `FullName`, `CountryCode`, `ADM1Code`, `ADM2Code`, `Lat`, `Long`, `FeatureID`) | Se repite el mismo set de 8 campos para 3 ubicaciones distintas: dónde está el Actor1, dónde está el Actor2, y dónde **ocurrió la acción** (`ActionGeo_*`, la que usamos como `Ubicacion` del evento). Pueden no coincidir con el país del actor (ej. presidente ruso de visita en Washington). `FeatureID` no siempre está disponible, por eso no sirve como llave natural. |
| **Gestión de datos** | `DATEADDED`, `SOURCEURL` | `DATEADDED` con resolución de 15 minutos; `SOURCEURL` es la URL/cita del primer artículo donde se detectó el evento (antecedente de nuestra tabla puente `Evento_Fuente`, que generaliza a *todas* las fuentes que lo reportan, no solo la primera). |

### 0.2) CAMEO Manual 1.1b3 — códigos de evento y de actor

**Códigos de evento (verbos):** La taxonomía tiene **20 categorías raíz** (2 dígitos, `EventRootCode`), cada una con subcódigos de 3 y 4 dígitos cada vez más específicos. Van desde cooperación hasta conflicto armado, por ejemplo: `01` Make Public Statement, `02` Appeal, `03` Express Intent to Cooperate, `04` Consult, `05`-`08` distintas formas de cooperación diplomática/material, `10` Demand, `11` Disapprove... hasta `17` Coerce, `18` Assault, `19` Fight y `20` Engage in Unconventional Mass Violence. Estas 20 categorías son justamente las que colapsan en las 4 `QuadClass`.

**Códigos de actor:** El `ActorCode` crudo de GDELT es la concatenación de bloques de 3 caracteres definidos por CAMEO, sin separadores fijos, por ejemplo `USAGOV` = país (`USA`) + rol primario (`GOV`). Los bloques posibles son:
- **País/región** (3 letras, código ONU) o un código internacional genérico cuando no aplica un país: `IGO` (organismo intergubernamental), `NGO` (ONG), `IMG` (grupo militarizado internacional), `MNC` (multinacional), `UIS` (actor estatal no identificado), `INT` (catch-all).
- **Rol primario** (el más usado): `GOV` Gobierno, `MIL` Militar, `OPP` Oposición política, `REB` Rebeldes, `INS`/`SEP` Insurgentes/separatistas, `COP` Policía, `JUD` Judicial, `SPY` Inteligencia, `UAF` Fuerzas armadas no alineadas.
- **Rol secundario** (especialidad, cuando no aplica un rol primario): `BUS` Empresa, `CRM` Criminal, `CVL` Civil, `MED` Medios, `EDU` Educación, `ELI` Elites, `LEG` Legislatura, `HRI` Derechos Humanos, entre otros.
- **Rol terciario**: `MOD` Moderado, `RAD` Radical (modificadores ambiguos como "extremista").
- Opcionalmente, bloques de **etnia**, **religión** (hasta 2 códigos) y un **código de organización específica** (ej. `IGOUNO` = Naciones Unidas dentro de la familia de IGOs).

Esto es lo que en GDELT 2.0 ya viene desagregado en columnas (`Type1/2/3Code`, `EthnicCode`, `Religion1/2Code`), y es la base de nuestra reclasificación de negocio `TipoActor` (Gobierno/ONG/Empresa/Otro) en la dimensión `Dim_Actor` del esquema estrella.

**Relevancia para el diseño:**
1. El `ActorCode` crudo **no debería partirse manualmente** en el modelo OLTP: GDELT ya provee los sub-campos separados, así que la entidad `Actor` los persiste tal cual (`CountryCode`, `KnownGroupCode`, `EthnicCode`, `Religion1/2Code`, `Type1/2/3Code`) en vez de intentar re-parsear el string concatenado.
2. `EventCode`/`EventBaseCode`/`EventRootCode` como texto (nunca entero) evita perder ceros a la izquierda — se va a reflejar en el esquema SQL de la sección 3.
3. La jerarquía de 20 categorías raíz + `QuadClass` es lo que permite responder la pregunta de negocio 4 (una cobertura puede re-etiquetarse con más de un `EventCode` a medida que avanza la historia) sin necesitar más que el catálogo `Codigo_CAMEO` / `Dim_CategoriaCAMEO` ya definido.

## 1) Base de datos operativa (OLTP) — Modelo Entidad-Relación

El sistema operativo de la redacción necesita registrar en tiempo real, cada evento de noticias detectado (a partir del feed de eventos GDELT 2.0), los actores que participan, la ubicación geográfica del hecho, las fuentes/medios que lo reportan y su clasificación CAMEO. Se diseñó el siguiente modelo relacional normalizado (3FN), pensado para inserciones/actualizaciones frecuentes (alta tasa de escritura, típica de un sistema OLTP) más que para agregaciones masivas.

### Entidades principales del modelo

- **Noticia_Evento**: Entidad central, un registro por evento detectado (equivalente a una fila del feed de eventos de GDELT).
- **Actor**: Personas, organizaciones, países o grupos que participan de un evento (Actor1 y Actor2 en la terminología CAMEO).
- **Ubicacion**: La geolocalización donde ocurrió la acción del evento (`ActionGeo_*` en GDELT).
- **Fuente_Medio**: Medio/dominio que publicó una noticia sobre el evento.
- **Codigo_CAMEO**: Catálogo de códigos de acción CAMEO (`EventCode`, `EventBaseCode`, `EventRootCode`, `QuadClass`) con su descripción.
- **Evento_Fuente**: Tabla puente que resuelve la relación M:N entre eventos y fuentes (un evento es reportado por varias fuentes, y una fuente reporta varios eventos — de ahí `NumSources` > 1 en GDELT).

### Llaves: PK, FK, compuestas y sustitutas (surrogate)

| Entidad | PK (surrogate) | Llave natural candidata | FK | Notas |
|---|---|---|---|---|
| `Noticia_Evento` | `EventoID` (INTEGER autoincrement) | `GlobalEventID` (se conserva como columna `UNIQUE NOT NULL`) | `Actor1ID`, `Actor2ID` → `Actor`; `CodigoCameoID` → `Codigo_CAMEO`; `ActionUbicacionID` → `Ubicacion` | Ver justificación abajo(*) |
| `Actor` | `ActorID` | `ActorCode` (`UNIQUE NOT NULL`) | — | `ActorCode` es un string CAMEO (ej. `USAGOV`), más largo y menos eficiente para indexar que un entero |
| `Ubicacion` | `UbicacionID` | `FeatureID` (candidata, puede ser nula) | — | GDELT no siempre provee `FeatureID`; no puede ser PK natural confiable |
| `Fuente_Medio` | `FuenteID` | `DominioURL` (`UNIQUE NOT NULL`) | — | El dominio es la unidad natural de "fuente" |
| `Codigo_CAMEO` | `CodigoCameoID` | `EventCode` (`UNIQUE NOT NULL`) | — | Catálogo maestro, cambia poco |
| `Evento_Fuente` | **Llave compuesta** por `EventoID`, `FuenteID` | — | `EventoID` → `Noticia_Evento`; `FuenteID` → `Fuente_Medio` | Tabla puente M:N; la combinación es naturalmente única |

**(*)¿Por qué usar una llave sustituta (`EventoID`) en lugar de la llave natural `GlobalEventID`?**

1. **Independencia del sistema de origen**: `GlobalEventID` es asignado por GDELT, un sistema externo. Si en algún momento se re-ingesta el mismo evento desde otra fuente (ej. corrección editorial, reproceso de un archivo, migración de proveedor de datos), la clave interna del medio no debe depender de la estabilidad de un identificador ajeno.
2. **Desacople de reglas de negocio**: Si GDELT cambiara su esquema de IDs entre versiones (como ya ocurrió entre GDELT 1.0 y 2.0), el modelo interno del medio permanece estable.
3. **Historial / control interno**: El surrogate permite versionar registros (ej. reprocesos, correcciones) sin colisionar con la clave natural, que se conserva como atributo único para trazabilidad y para el ETL hacia el almacén de datos.

### Tipos de relación

| Relación | Cardinalidad | Opcionalidad | Justificación |
|---|---|---|---|
| `Noticia_Evento` → `Actor` (Actor1) | N:1 | **Obligatoria** | Todo evento CAMEO tiene al menos un actor iniciador |
| `Noticia_Evento` → `Actor` (Actor2) | N:1 | **Opcional** | Muchos eventos son acciones unilaterales sin un segundo actor (`Actor2CountryCode` suele venir vacío en GDELT) |
| `Noticia_Evento` → `Ubicacion` | N:1 | **Opcional** | La geolocalización de la acción (`ActionGeo_*`) puede no resolverse |
| `Noticia_Evento` → `Codigo_CAMEO` | N:1 | **Obligatoria** | Todo evento debe clasificarse con un código de acción CAMEO |
| `Noticia_Evento` ↔ `Fuente_Medio` | **M:N** (vía `Evento_Fuente`) | Obligatoria del lado de `Noticia_Evento` (`NumSources` ≥ 1) | Un evento es cubierto por múltiples medios y un medio cubre múltiples eventos; esto se resuelve con tabla puente porque no es 1:N en ningún sentido |

No se identificaron relaciones 1:1 en este modelo: todas las asociaciones entre entidades operativas del dominio periodístico son de tipo N:1 o M:N, ya que ninguna entidad tiene una correspondencia exclusiva de "uno a uno" con otra (un actor participa de muchos eventos, un evento puede tener dos actores distintos, etc.).

### Diagrama
Ver la página **"OLTP - Modelo Entidad-Relacion"** del archivo [`TP2.drawio`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP2.drawio).

## 2) Modelo dimensional (OLAP) — Esquema Estrella

Para responder las preguntas de negocio de Inteligencia de Contenidos se diseñó un **esquema en estrella** con grano "un evento periodístico" (mismo nivel de detalle que `Noticia_Evento` en el OLTP, para simplificar el ETL), desnormalizando los atributos descriptivos de actores, ubicación y fuente en dimensiones, y agregando una dimensión temporal enriquecida y una tabla puente para las categorías CAMEO múltiples.

### Diagrama

Ver la página **"OLAP - Esquema Estrella"** del archivo [`TP2.drawio`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP2.drawio).

### Tabla de hechos: `Fact_Cobertura_Mediatica`

**Métricas (medidas aditivas):**
- `NumMentions`: Cantidad de menciones del evento en todos los artículos monitoreados.
- `NumSources`: Cantidad de fuentes/medios distintos que lo reportan.
- `NumArticles`: Cantidad total de artículos que lo mencionan.
- `AvgTone`: Tono promedio de la cobertura (semi-aditiva: se pondera por cantidad de eventos al agregar).
- `GoldsteinScale`: Puntaje de impacto/estabilidad del evento (semi-aditiva).
- `CantidadEventos`: Medida de conteo (=1 por fila) para poder responder "cantidad de eventos" en cualquier agregación (pregunta de negocio 3).

**Dimensiones asociadas:** `Dim_Tiempo`, `Dim_Actor` (rol Actor1), `Dim_Ubicacion`, `Dim_Fuente`, y (vía tabla puente) `Dim_CategoriaCAMEO`.

### Dimensión temporal: `Dim_Tiempo`

| Atributo | Descripción |
|---|---|
| `Fecha` | Fecha natural del evento (`YYYY-MM-DD`), llave natural única |
| `Anio` | Año calendario |
| `Trimestre` | 1 a 4 |
| `Mes` | 1 a 12 |
| `Dia_Semana` | Nombre del día (Lunes…Domingo) |
| `Es_Fin_De_Semana` | Booleano, `True` si `Dia_Semana` ∈ {Sábado, Domingo} — responde directamente a la pregunta de negocio 3 |
| `Es_Festivo` | Booleano, contra un calendario de feriados cargado aparte — responde a la pregunta de negocio 3 |

### Tabla puente: `Bridge_Evento_Categoria`

El feed crudo de eventos de GDELT asigna **un único** `EventCode` por evento (`Noticia_Evento.CodigoCameoID` en el OLTP). Sin embargo, en la práctica editorial una misma cobertura periodística de un evento en desarrollo suele reclasificarse o etiquetarse bajo **múltiples** categorías CAMEO a medida que se publican notas relacionadas (ej. un mismo `GlobalEventID` con notas de seguimiento tipificadas como "042 - Appeal" y luego "111 - Demonstrate/Protest"). Para modelar esa relación **M:N entre un evento y sus categorías temáticas** sin duplicar las métricas de la tabla de hechos (lo que rompería la aditividad de `NumMentions`/`NumArticles`), se agrega:

- `Bridge_Evento_Categoria (EventoID, CategoriaID, Peso_Ponderacion)`
- `Peso_Ponderacion` = `1 / n_categorias_del_evento`, para poder distribuir proporcionalmente una medida aditiva al agregar por categoría sin sobre-contar (patrón estándar de *weighting factor* en tablas puente de Kimball).

Esto responde directamente a la pregunta de negocio 4: alcanza con `GROUP BY CategoriaID HAVING COUNT(DISTINCT EventoID)` vía la tabla puente, o `GROUP BY EventoID HAVING COUNT(DISTINCT CategoriaID) > 1` para listar los eventos con clasificación multi-temática.

## 3) Implementación con SQL (opcional) — SQLite

> Esta sección fue co-realizada con Claude

Se implementan ambos modelos (OLTP y OLAP) en dos bases SQLite independientes (`tp2_oltp.db` y `tp2_olap.db`), se cargan datos sintéticos de ejemplo (con la misma forma que produciría un ETL desde el feed real de GDELT) y se responden las 4 preguntas de negocio con consultas SQL sobre el almacén dimensional.

### 3.1) Creación del esquema OLTP

In [1]:
import sqlite3
import os

DB_OLTP = "tp2_oltp.db"
if os.path.exists(DB_OLTP):
    os.remove(DB_OLTP)

con_oltp = sqlite3.connect(DB_OLTP)
cur = con_oltp.cursor()
cur.executescript('''
CREATE TABLE Actor (
    ActorID INTEGER PRIMARY KEY AUTOINCREMENT,
    ActorCode TEXT UNIQUE NOT NULL,
    Nombre TEXT,
    CountryCode TEXT,
    KnownGroupCode TEXT,
    EthnicCode TEXT,
    Religion1Code TEXT,
    Religion2Code TEXT,
    Type1Code TEXT,
    Type2Code TEXT,
    Type3Code TEXT
);

CREATE TABLE Ubicacion (
    UbicacionID INTEGER PRIMARY KEY AUTOINCREMENT,
    GeoType INTEGER,
    FullName TEXT,
    CountryCode TEXT,
    ADM1Code TEXT,
    ADM2Code TEXT,
    Lat REAL,
    Long REAL,
    FeatureID TEXT
);

CREATE TABLE Codigo_CAMEO (
    CodigoCameoID INTEGER PRIMARY KEY AUTOINCREMENT,
    EventCode TEXT UNIQUE NOT NULL,
    EventBaseCode TEXT,
    EventRootCode TEXT,
    QuadClass INTEGER,
    Descripcion TEXT
);

CREATE TABLE Fuente_Medio (
    FuenteID INTEGER PRIMARY KEY AUTOINCREMENT,
    DominioURL TEXT UNIQUE NOT NULL,
    NombreMedio TEXT,
    PaisMedio TEXT,
    TipoMedio TEXT
);

CREATE TABLE Noticia_Evento (
    EventoID INTEGER PRIMARY KEY AUTOINCREMENT,
    GlobalEventID INTEGER UNIQUE NOT NULL,
    FechaEvento TEXT NOT NULL,
    MonthYear INTEGER,
    Year INTEGER,
    FractionDate REAL,
    Actor1ID INTEGER NOT NULL REFERENCES Actor(ActorID),
    Actor2ID INTEGER REFERENCES Actor(ActorID),
    CodigoCameoID INTEGER NOT NULL REFERENCES Codigo_CAMEO(CodigoCameoID),
    ActionUbicacionID INTEGER REFERENCES Ubicacion(UbicacionID),
    IsRootEvent INTEGER,
    GoldsteinScale REAL,
    AvgTone REAL,
    NumMentions INTEGER,
    NumSources INTEGER,
    NumArticles INTEGER,
    DATEADDED TEXT
);

CREATE TABLE Evento_Fuente (
    EventoID INTEGER NOT NULL REFERENCES Noticia_Evento(EventoID),
    FuenteID INTEGER NOT NULL REFERENCES Fuente_Medio(FuenteID),
    SourceURL TEXT,
    FechaPublicacion TEXT,
    PRIMARY KEY (EventoID, FuenteID)
);
''')
con_oltp.commit()
print("Esquema OLTP creado en", DB_OLTP)

Esquema OLTP creado en tp2_oltp.db


### 3.2) Carga de datos de ejemplo (simulando un lote ingerido desde GDELT)

In [2]:
# Actores (código CAMEO real: tipo de actor embebido en Type1Code)
actores = [
    ("USAGOV", "United States Government", "USA", None, None, None, None, "GOV", None, None),
    ("CHNGOV", "China Government", "CHN", None, None, None, None, "GOV", None, None),
    ("NGOAMN", "Amnesty International", None, "AMN", None, None, None, "NGO", None, None),
    ("BUSTEC", "Tech Corp", None, None, None, None, None, "BUS", None, None),
    ("IGOUNO", "United Nations", None, "UNO", None, None, None, "IGO", None, None),
    ("ARGGOV", "Argentina Government", "ARG", None, None, None, None, "GOV", None, None),
]

cur.executemany(
    "INSERT INTO Actor (ActorCode, Nombre, CountryCode, KnownGroupCode, EthnicCode, "
    "Religion1Code, Religion2Code, Type1Code, Type2Code, Type3Code) VALUES (?,?,?,?,?,?,?,?,?,?)",
    actores,
)

ubicaciones = [
    (4, "Buenos Aires, Argentina", "ARG", None, None, -34.61, -58.38, "AR01"),
    (4, "Washington, United States", "USA", None, None, 38.90, -77.04, "US01"),
    (4, "Beijing, China", "CHN", None, None, 39.90, 116.40, "CH01"),
    (4, "Geneva, Switzerland", "CHE", None, None, 46.20, 6.14, "SW01"),
]
cur.executemany(
    "INSERT INTO Ubicacion (GeoType, FullName, CountryCode, ADM1Code, ADM2Code, Lat, Long, FeatureID) "
    "VALUES (?,?,?,?,?,?,?,?)",
    ubicaciones,
)

# Códigos CAMEO (subset real del Event Codebook V2.0)
cameo_codes = [
    ("010", "01", "01", 1, "Make statement, not specified below"),
    ("042", "04", "04", 1, "Make a visit"),
    ("111", "11", "11", 3, "Demonstrate or rally"),
    ("172", "17", "17", 4, "Impose administrative sanctions"),
    ("040", "04", "04", 1, "Consult, not specified below"),
]
cur.executemany(
    "INSERT INTO Codigo_CAMEO (EventCode, EventBaseCode, EventRootCode, QuadClass, Descripcion) VALUES (?,?,?,?,?)",
    cameo_codes,
)

fuentes = [
    ("reuters.com", "Reuters", "GBR", "Agencia"),
    ("bbc.com", "BBC News", "GBR", "Broadcast"),
    ("clarin.com", "Clarín", "ARG", "Diario"),
    ("xinhuanet.com", "Xinhua", "CHN", "Agencia"),
    ("apnews.com", "Associated Press", "USA", "Agencia"),
]
cur.executemany(
    "INSERT INTO Fuente_Medio (DominioURL, NombreMedio, PaisMedio, TipoMedio) VALUES (?,?,?,?)",
    fuentes,
)
con_oltp.commit()
print("Catálogos (Actor, Ubicacion, Codigo_CAMEO, Fuente_Medio) cargados.")

Catálogos (Actor, Ubicacion, Codigo_CAMEO, Fuente_Medio) cargados.


In [3]:
from datetime import date

# Eventos de ejemplo: (GlobalEventID, FechaEvento, Actor1Code, Actor2Code, EventCode,
#                       UbicacionIdx(1-based), IsRootEvent, GoldsteinScale, AvgTone,
#                       NumMentions, NumSources, NumArticles)
eventos_raw = [
    (900001, date(2026, 9, 7),  "ARGGOV", None,     "010", 1, 1, 1.9,  1.2, 12, 3, 14),   # lunes
    (900002, date(2026, 9, 7),  "USAGOV", "CHNGOV", "042", 2, 1, 3.4,  0.8,  8, 2,  9),   # lunes
    (900003, date(2026, 9, 8),  "NGOAMN", "USAGOV", "111", 2, 0, -4.0, -3.1, 25, 4, 30),  # martes
    (900004, date(2026, 9, 12), "BUSTEC", None,     "010", 2, 1, 1.0,  2.5,  5, 1,  5),   # sábado (fin de semana)
    (900005, date(2026, 9, 13), "IGOUNO", "CHNGOV", "172", 4, 1, -6.5, -5.4, 18, 3, 22),  # domingo (fin de semana)
    (900006, date(2026, 12, 25),"USAGOV", None,     "040", 2, 1, 2.0,  1.5,  6, 2,  7),   # feriado (Navidad)
    (900007, date(2026, 9, 9),  "CHNGOV", "IGOUNO", "111", 3, 0, -3.2, -2.0, 30, 5, 35),  # miércoles
]

actor_id = {row[0]: idx + 1 for idx, row in enumerate(actores)}  # ActorCode -> ActorID (orden de insercion)
cameo_id = {row[0]: idx + 1 for idx, row in enumerate(cameo_codes)}  # EventCode -> CodigoCameoID

for gid, fecha, a1, a2, ecode, ubic_idx, is_root, goldstein, tone, mentions, sources, articles in eventos_raw:
    cur.execute(
        "INSERT INTO Noticia_Evento (GlobalEventID, FechaEvento, MonthYear, Year, FractionDate, "
        "Actor1ID, Actor2ID, CodigoCameoID, ActionUbicacionID, IsRootEvent, GoldsteinScale, AvgTone, "
        "NumMentions, NumSources, NumArticles, DATEADDED) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        (
            gid, fecha.isoformat(), int(fecha.strftime("%Y%m")), fecha.year, fecha.year + fecha.timetuple().tm_yday / 365,
            actor_id[a1], actor_id[a2] if a2 else None, cameo_id[ecode], ubic_idx, is_root, goldstein, tone,
            mentions, sources, articles, fecha.isoformat(),
        ),
    )
con_oltp.commit()

evento_id = {gid: idx + 1 for idx, (gid, *_r) in enumerate(eventos_raw)}

# Evento_Fuente: cada evento cubierto por NumSources fuentes distintas (tabla puente M:N)
evento_fuente_rows = [
    (evento_id[900001], 3, "https://clarin.com/n1", "2026-09-07"),
    (evento_id[900001], 1, "https://reuters.com/n1", "2026-09-07"),
    (evento_id[900001], 5, "https://apnews.com/n1", "2026-09-07"),
    (evento_id[900002], 1, "https://reuters.com/n2", "2026-09-07"),
    (evento_id[900002], 4, "https://xinhuanet.com/n2", "2026-09-07"),
    (evento_id[900003], 2, "https://bbc.com/n3", "2026-09-08"),
    (evento_id[900003], 5, "https://apnews.com/n3", "2026-09-08"),
    (evento_id[900004], 1, "https://reuters.com/n4", "2026-09-12"),
    (evento_id[900005], 2, "https://bbc.com/n5", "2026-09-13"),
    (evento_id[900005], 3, "https://clarin.com/n5", "2026-09-13"),
    (evento_id[900006], 5, "https://apnews.com/n6", "2026-12-25"),
    (evento_id[900007], 4, "https://xinhuanet.com/n7", "2026-09-09"),
    (evento_id[900007], 2, "https://bbc.com/n7", "2026-09-09"),
]
cur.executemany(
    "INSERT INTO Evento_Fuente (EventoID, FuenteID, SourceURL, FechaPublicacion) VALUES (?,?,?,?)",
    evento_fuente_rows,
)
con_oltp.commit()
print(f"{len(eventos_raw)} eventos y {len(evento_fuente_rows)} vínculos Evento_Fuente cargados en OLTP.")

7 eventos y 13 vínculos Evento_Fuente cargados en OLTP.


### 3.3) Creación del esquema OLAP (Esquema Estrella) y ETL desde el OLTP

In [4]:
DB_OLAP = "tp2_olap.db"
if os.path.exists(DB_OLAP):
    os.remove(DB_OLAP)

con_olap = sqlite3.connect(DB_OLAP)
cur_olap = con_olap.cursor()
cur_olap.executescript('''
CREATE TABLE Dim_Tiempo (
    TiempoID INTEGER PRIMARY KEY AUTOINCREMENT,
    Fecha TEXT UNIQUE NOT NULL,
    Anio INTEGER,
    Trimestre INTEGER,
    Mes INTEGER,
    Dia_Semana TEXT,
    Es_Fin_De_Semana INTEGER,
    Es_Festivo INTEGER
);

CREATE TABLE Dim_Actor (
    ActorID INTEGER PRIMARY KEY AUTOINCREMENT,
    ActorCode TEXT UNIQUE,
    Nombre TEXT,
    CountryCode TEXT,
    TipoActor TEXT
);

CREATE TABLE Dim_Ubicacion (
    UbicacionID INTEGER PRIMARY KEY AUTOINCREMENT,
    Pais TEXT,
    ADM1 TEXT,
    ADM2 TEXT,
    Lat REAL,
    Long REAL,
    TipoGeo INTEGER
);

CREATE TABLE Dim_Fuente (
    FuenteID INTEGER PRIMARY KEY AUTOINCREMENT,
    Dominio TEXT UNIQUE,
    NombreMedio TEXT,
    PaisMedio TEXT
);

CREATE TABLE Dim_CategoriaCAMEO (
    CategoriaID INTEGER PRIMARY KEY AUTOINCREMENT,
    EventCode TEXT UNIQUE,
    EventBaseCode TEXT,
    EventRootCode TEXT,
    QuadClass INTEGER,
    Descripcion TEXT
);

CREATE TABLE Fact_Cobertura_Mediatica (
    EventoID INTEGER PRIMARY KEY,
    GlobalEventID INTEGER UNIQUE NOT NULL,
    TiempoID INTEGER REFERENCES Dim_Tiempo(TiempoID),
    Actor1ID INTEGER REFERENCES Dim_Actor(ActorID),
    UbicacionID INTEGER REFERENCES Dim_Ubicacion(UbicacionID),
    FuenteID INTEGER REFERENCES Dim_Fuente(FuenteID),
    NumMentions INTEGER,
    NumSources INTEGER,
    NumArticles INTEGER,
    AvgTone REAL,
    GoldsteinScale REAL,
    CantidadEventos INTEGER DEFAULT 1
);

CREATE TABLE Bridge_Evento_Categoria (
    EventoID INTEGER NOT NULL REFERENCES Fact_Cobertura_Mediatica(EventoID),
    CategoriaID INTEGER NOT NULL REFERENCES Dim_CategoriaCAMEO(CategoriaID),
    Peso_Ponderacion REAL,
    PRIMARY KEY (EventoID, CategoriaID)
);
''')
con_olap.commit()
print("Esquema OLAP (estrella) creado en", DB_OLAP)

Esquema OLAP (estrella) creado en tp2_olap.db


In [5]:
import calendar

# --- Dim_Actor: se reclasifica Type1Code -> TipoActor de negocio ---
TIPO_ACTOR = {"GOV": "Gobierno", "NGO": "ONG", "IGO": "ONG", "BUS": "Empresa"}

cur_olap.executemany(
    "INSERT INTO Dim_Actor (ActorCode, Nombre, CountryCode, TipoActor) VALUES (?,?,?,?)",
    [(code_, nombre, country, TIPO_ACTOR.get(type1, "Otro"))
     for code_, nombre, country, _, _, _, _, type1, _, _ in actores],
)

# --- Dim_Ubicacion ---
cur_olap.executemany(
    "INSERT INTO Dim_Ubicacion (Pais, ADM1, ADM2, Lat, Long, TipoGeo) VALUES (?,?,?,?,?,?)",
    [(full.split(",")[-1].strip(), None, None, lat, lon, geo)
     for geo, full, country, adm1, adm2, lat, lon, feat in ubicaciones],
)

# --- Dim_Fuente ---
cur_olap.executemany(
    "INSERT INTO Dim_Fuente (Dominio, NombreMedio, PaisMedio) VALUES (?,?,?)",
    [(dominio, nombre, pais) for dominio, nombre, pais, tipo in fuentes],
)

# --- Dim_CategoriaCAMEO ---
cur_olap.executemany(
    "INSERT INTO Dim_CategoriaCAMEO (EventCode, EventBaseCode, EventRootCode, QuadClass, Descripcion) VALUES (?,?,?,?,?)",
    cameo_codes,
)
con_olap.commit()

# --- Dim_Tiempo: una fila por fecha distinta usada por los eventos ---
FERIADOS = {date(2026, 12, 25)}  # calendario de feriados (ejemplo: Navidad)

fechas = sorted({fecha for _gid, fecha, *_r in eventos_raw})
for f in fechas:
    dia_semana_es = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"][f.weekday()]
    es_finde = f.weekday() >= 5
    es_festivo = f in FERIADOS
    cur_olap.execute(
        "INSERT INTO Dim_Tiempo (Fecha, Anio, Trimestre, Mes, Dia_Semana, Es_Fin_De_Semana, Es_Festivo) "
        "VALUES (?,?,?,?,?,?,?)",
        (f.isoformat(), f.year, (f.month - 1) // 3 + 1, f.month, dia_semana_es, int(es_finde), int(es_festivo)),
    )
con_olap.commit()
print("Dimensiones cargadas: Dim_Actor, Dim_Ubicacion, Dim_Fuente, Dim_CategoriaCAMEO, Dim_Tiempo.")

Dimensiones cargadas: Dim_Actor, Dim_Ubicacion, Dim_Fuente, Dim_CategoriaCAMEO, Dim_Tiempo.


In [6]:
tiempo_id = {f: idx + 1 for idx, f in enumerate(fechas)}
actor_olap_id = {row[0]: idx + 1 for idx, row in enumerate(actores)}
ubic_olap_id = {idx + 1: idx + 1 for idx in range(len(ubicaciones))}  # mismo orden de carga
fuente_olap_id = {row[0]: idx + 1 for idx, row in enumerate(fuentes)}
cameo_olap_id = {row[0]: idx + 1 for idx, row in enumerate(cameo_codes)}

# --- Fact_Cobertura_Mediatica: 1 fila por evento (mismo grano que Noticia_Evento) ---
for gid, fecha, a1, a2, ecode, ubic_idx, is_root, goldstein, tone, mentions, sources, articles in eventos_raw:
    cur_olap.execute(
        "INSERT INTO Fact_Cobertura_Mediatica (EventoID, GlobalEventID, TiempoID, Actor1ID, UbicacionID, "
        "FuenteID, NumMentions, NumSources, NumArticles, AvgTone, GoldsteinScale, CantidadEventos) "
        "VALUES (?,?,?,?,?,?,?,?,?,?,?,1)",
        (
            evento_id[gid], gid, tiempo_id[fecha], actor_olap_id[a1], ubic_olap_id[ubic_idx],
            None, mentions, sources, articles, tone, goldstein,
        ),
    )

# --- Bridge_Evento_Categoria: eventos con clasificación multi-temática ---
# Se simula que algunas coberturas se re-etiquetaron con más de un código CAMEO
# a medida que se publicaron notas de seguimiento del mismo GlobalEventID.
multi_categoria = {
    900001: ["010"],
    900002: ["042", "040"],          # visita + consulta diplomática -> multi-tema
    900003: ["111", "172"],          # protesta + sanciones administrativas -> multi-tema
    900004: ["010"],
    900005: ["172"],
    900006: ["040"],
    900007: ["111"],
}
for gid, codigos in multi_categoria.items():
    peso = round(1 / len(codigos), 4)
    for c in codigos:
        cur_olap.execute(
            "INSERT INTO Bridge_Evento_Categoria (EventoID, CategoriaID, Peso_Ponderacion) VALUES (?,?,?)",
            (evento_id[gid], cameo_olap_id[c], peso),
        )
con_olap.commit()
print("Fact_Cobertura_Mediatica y Bridge_Evento_Categoria cargados.")

Fact_Cobertura_Mediatica y Bridge_Evento_Categoria cargados.


### 3.4) Consultas de negocio sobre el almacén dimensional

Se responden las 4 preguntas clave directamente contra el esquema estrella (`Fact_Cobertura_Mediatica` + dimensiones).

**Pregunta 1 — Rendimiento e Impacto:** volumen de menciones y artículos por país y mes.

In [7]:
import pandas as pd

q1 = pd.read_sql_query('''
    SELECT du.Pais, dt.Mes,
           SUM(f.NumMentions) AS Total_Menciones,
           SUM(f.NumArticles) AS Total_Articulos
    FROM Fact_Cobertura_Mediatica f
    JOIN Dim_Ubicacion du ON f.UbicacionID = du.UbicacionID
    JOIN Dim_Tiempo dt ON f.TiempoID = dt.TiempoID
    GROUP BY du.Pais, dt.Mes
    ORDER BY Total_Menciones DESC
''', con_olap)
q1

,Pais,Mes,Total_Menciones,Total_Articulos
0,United States,9,38,44
1,China,9,30,35
2,Switzerland,9,18,22
3,Argentina,9,12,14
4,United States,12,6,7


**Pregunta 2 — Sentimiento de la Cobertura:** tono promedio (`AvgTone`) por tipo de actor.

In [8]:
q2 = pd.read_sql_query('''
    SELECT da.TipoActor,
           ROUND(AVG(f.AvgTone), 3) AS AvgTone_Promedio,
           COUNT(*) AS Cantidad_Eventos
    FROM Fact_Cobertura_Mediatica f
    JOIN Dim_Actor da ON f.Actor1ID = da.ActorID
    GROUP BY da.TipoActor
    ORDER BY AvgTone_Promedio ASC
''', con_olap)
q2

,TipoActor,AvgTone_Promedio,Cantidad_Eventos
0,ONG,-4.250,2
1,Gobierno,0.375,4
2,Empresa,2.500,1


**Pregunta 3 — Frecuencia y Temporalidad:** intensidad del flujo entre días hábiles, fines de semana y feriados.

In [9]:
q3 = pd.read_sql_query('''
    SELECT
        CASE
            WHEN dt.Es_Festivo = 1 THEN 'Feriado'
            WHEN dt.Es_Fin_De_Semana = 1 THEN 'Fin de semana'
            ELSE 'Día hábil'
        END AS Tipo_Dia,
        COUNT(*) AS Cantidad_Eventos,
        ROUND(AVG(f.NumArticles), 2) AS Promedio_Articulos
    FROM Fact_Cobertura_Mediatica f
    JOIN Dim_Tiempo dt ON f.TiempoID = dt.TiempoID
    GROUP BY Tipo_Dia
    ORDER BY Cantidad_Eventos DESC
''', con_olap)
q3

,Tipo_Dia,Cantidad_Eventos,Promedio_Articulos
0,Día hábil,4,22.0
1,Fin de semana,2,13.5
2,Feriado,1,7.0


**Pregunta 4 — Cobertura Multi-Temática:** eventos clasificados bajo más de un código CAMEO simultáneamente (vía `Bridge_Evento_Categoria`).

In [10]:
q4 = pd.read_sql_query('''
    SELECT f.GlobalEventID,
           COUNT(DISTINCT b.CategoriaID) AS Cantidad_Categorias,
           GROUP_CONCAT(DISTINCT dc.EventCode) AS Codigos_CAMEO
    FROM Bridge_Evento_Categoria b
    JOIN Fact_Cobertura_Mediatica f ON b.EventoID = f.EventoID
    JOIN Dim_CategoriaCAMEO dc ON b.CategoriaID = dc.CategoriaID
    GROUP BY f.GlobalEventID
    HAVING Cantidad_Categorias > 1
    ORDER BY Cantidad_Categorias DESC
''', con_olap)
q4

,GlobalEventID,Cantidad_Categorias,Codigos_CAMEO
0,900003,2,"111,172"
1,900002,2,"042,040"


In [11]:
con_oltp.close()
con_olap.close()
print("Conexiones cerradas. Bases generadas:", DB_OLTP, "y", DB_OLAP)

Conexiones cerradas. Bases generadas: tp2_oltp.db y tp2_olap.db


---